In [4]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from src.context_builder import build_context
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

### Testing Groq API

In [5]:
#adding Groq API
load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"))

In [5]:
#testing that LLM works with easy query
response = llm.invoke("What is a good gaming headset?")
print(response.content)

There are many great gaming headsets available, depending on your budget, gaming platform (e.g., PC, Xbox, PlayStation, or Nintendo Switch), and personal preferences. Here are some popular options:

**Budget-friendly (Under $100)**

1. HyperX Cloud II: A popular, affordable headset with excellent sound quality and a comfortable design.
2. Turtle Beach Recon 200: A budget-friendly option with a strong focus on comfort and a sleek design.
3. SteelSeries Arctis 3: A lightweight, foldable headset with good sound quality and a stylish design.

**Mid-range ($100-$250)**

1. SteelSeries Arctis 7: A high-end version of the Arctis 3, with wireless connectivity and a more premium design.
2. HyperX Cloud Flight: A wireless headset with long battery life and excellent sound quality.
3. Astro Gaming A40: A high-end headset with excellent sound quality and a comfortable design.

**High-end ($250-$500)**

1. Sennheiser GSP 670: A premium headset with excellent sound quality, a comfortable design, and

### Prompt exploration: Semantic RAG

In [6]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local('../data/processed/faiss_index', embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
#prompt testing

#trying with a less detailed prompt, minimal instructions
prompt_1 = ChatPromptTemplate.from_template("""
You are a helpful Amazon video game shopping assistant.
Answer using ONLY the context. Be concise and relevant with your product recommendations.

Context: {context}
Question: {input}
Answer:
""")

#more detailed prompt of exactly what to make product recommendations on
prompt_2 = ChatPromptTemplate.from_template("""
You are a helpful Amazon video game product shopping assistant.
Answer the question using ONLY the information in the context given.
The context contains real Amazon product reviews and metadata (product title, rating, price, description etc.).

- Recommend specific products that match the user's query
- Support your recommendation with concise and relevant evidence from the reviews and ratings
- Consider important product aspects like price and rating
- Only say you don't know if the context truly does not contain relevant information

Context: {context}
Question: {input}
Answer:
""")

In [16]:
#testing pipeline with different prompts on medium-level query 

query = "What is a beginner friendly game for ps4?"

for i, p in enumerate([prompt_1, prompt_2], 1):
    chain = (
        {"context": retriever | build_context, "input": RunnablePassthrough()}
        | p
        | llm
        | StrOutputParser()
    )
    print(f"\n{'='*60}\nPrompt {i}\n{'='*60}") #adding divider to output for easier prompt comparison
    print(chain.invoke(query))


Prompt 1
Based on the reviews, I would recommend "Forager" for a beginner-friendly game on PS4. It has a high rating of 4.6/5 and is described as a fun, simple, and relaxing game that's great for the entire family. The review mentions that it's "addictive" and that the player's grandson loves it, so it's likely to be engaging and easy to pick up for beginners.

Prompt 2
Based on the provided reviews and metadata, I recommend "Forager" for PS4 as a beginner-friendly game. 

Evidence from the review supports this recommendation: 

- The reviewer states that "this game is simply amazing it is quite fun for the entire family" which implies that it's suitable for players of different skill levels, including beginners.
- They also mention that it's a "great game for the PS4 family friendly and fun" which further supports its beginner-friendliness.
- Additionally, the game's description mentions that it "starts small and improves your base skills equipment network of friends and enemies" whi

In [9]:
# prompt 3 - having LLM recommend multiple products

prompt_3 = ChatPromptTemplate.from_template("""
You are a helpful Amazon video game product shopping assistant.
Answer the question using ONLY the information in the context given.
The context contains real Amazon product reviews and metadata (product title, rating, price, description etc.).

- Recommend up to 3 products that match the user's query
- Support your recommendation with evidence from the reviews and ratings
- Consider important product aspects like price and rating
- Only say you don't know if the context truly does not contain relevant information

Context: {context}
Question: {input}
Answer:
""")

In [11]:
query2 = "What are some fun video games for sports lovers?"

chain2 = (
    {
        "context": retriever | build_context,
        "input": RunnablePassthrough()
    }
    | prompt_3
    | llm
    | StrOutputParser()
    )

print(f"\n{'='*60}\nPrompt 3\n{'='*60}")
print(chain2.invoke(query2))


Prompt 3
Based on the context you've provided, I'd be happy to recommend some fun video games for sports lovers. Here are three options that match your query:

1. **Instant Sports Plus (Nintendo Switch)**: This game has a rating of 4.3/5 and is described as the most complete experience of the Instant Sports series, with 15 different sports and activities to enjoy. The reviewer "basic game play" found it to be a "cute" game, and another reviewer mentioned it's "fun for my 10yr old." Since it supports up to 4 local players, it's perfect for multiplayer fun. (Rating: 4.3/5, Price: Unknown)

2. **Sports Party (Nintendo Switch)**: Although this game received a mixed review from one user, who was looking for a motion movement game, it still has a rating of 4.2/5. The reviewer mentioned it's "fun" and their 8-year-old grandson loves it. It's a great option if you're looking for a game that's accessible to anyone. (Rating: 4.2/5, Price: $24.08)

3. **Instant Sports Allstars for Nintendo Switc

In [12]:
#what if the user asks for 1 product, will LLM recognize this or will it recommend 3 regardless?
query3 = 'Xbox controller with the best battery life'

print(chain2.invoke(query3))

Based on the context provided, I recommend the following Xbox controller products with the best battery life:

1. **Venom Xbox Controller Rechargeable Battery Twin Pack**:
This product has a 4.4/5 rating and is priced at $26.24. According to the reviews, it has "decent battery life" and can last up to 15 hours from a full charge. Although one reviewer mentioned that one of the batteries died quickly, the overall reviews suggest that this product has good battery life.

2. **Xbox Controller with 1400mAh Lithium Battery**:
This product has a 4.4/5 rating and is priced at $39.98. Although one reviewer mentioned that the buttons are misplaced, there are no comments about the battery life. However, the product description mentions a 1400mAh lithium battery, which suggests that it has a decent battery life.

3. **Theocleo Black 3600mAh Replacement Battery for Xbox 360 Controller**:
This product has a 5.0/5 rating and is priced at an unknown price (since the price is not mentioned in the cont

**Selected prompt**: Prompt 2 - See rational in [milestone2_discussion.md](https://github.com/UBC-MDS/DSCI_575_project_nourabd_mailysg/blob/main/results/milestone2_discussion.md)